# The Agent Loop: Raw, No Dependencies

**Phase 04** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-04/01-the-agent-loop.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Lesson 04-01: The Agent Loop: Raw, No Dependencies
Phase 04: Agents - Patterns That Survive Production

A complete agent loop in ~120 lines using only the anthropic SDK.
The loop: take a user goal, call Claude with tool schemas, check for
tool_use, execute tools, send tool_result back, repeat until end_turn.
"""

import anthropic
import json
import math

client = anthropic.Anthropic()

### Tool definitions (JSON Schema format, passed directly to the API)

In [ ]:
TOOLS = [
    {
        "name": "calculator",
        "description": (
            "Evaluate a mathematical expression. "
            "Input must be a valid Python math expression using standard operators "
            "and the math module (e.g. math.sqrt, math.pi)."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "A Python-evaluable math expression, e.g. '144 * 17' or 'math.sqrt(144)'"
                }
            },
            "required": ["expression"]
        }
    },
    {
        "name": "web_search",
        "description": (
            "Search the web for current information. "
            "Use this for facts that may have changed recently or are not in training data."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query string"
                }
            },
            "required": ["query"]
        }
    }
]

### Tool implementations

In [ ]:
def run_calculator(expression: str) -> str:
    """Safely evaluate a math expression."""
    try:
        # Restrict eval: only math module, no builtins
        result = eval(expression, {"__builtins__": {}, "math": math})
        return str(result)
    except Exception as e:
        return f"Calculator error: {e}"


def run_web_search(query: str) -> str:
    """Stub web search. Replace with a real search API in production."""
    return (
        f"[STUB] Search results for '{query}':\n"
        f"1. Example result: placeholder information about {query}.\n"
        f"2. Note: replace this stub with a real search API (e.g. Brave, Serper, Tavily)."
    )


# Tool registry: name -> callable(args_dict) -> str
TOOL_REGISTRY = {
    "calculator": lambda args: run_calculator(args["expression"]),
    "web_search": lambda args: run_web_search(args["query"]),
}


def execute_tools(tool_use_blocks: list) -> list[dict]:
    """
    Execute all tool_use content blocks and return a list of tool_result dicts.
    These go into a user message as: {"role": "user", "content": tool_results}
    """
    results = []
    for block in tool_use_blocks:
        tool_name = block.name
        tool_input = block.input
        tool_use_id = block.id

        if tool_name in TOOL_REGISTRY:
            output = TOOL_REGISTRY[tool_name](tool_input)
        else:
            output = f"Error: unknown tool '{tool_name}'"

        results.append({
            "type": "tool_result",
            "tool_use_id": tool_use_id,
            "content": output
        })
    return results

### The agent loop

In [ ]:
def run_agent(goal: str, max_iterations: int = 10, verbose: bool = True) -> str:
    """
    Run an agent loop until stop_reason is end_turn or max_iterations is reached.

    Message history grows by 2 per tool-use turn:
      - assistant message (with tool_use blocks)
      - user message (with tool_result blocks)
    """
    messages = [{"role": "user", "content": goal}]
    system = (
        "You are a precise assistant with access to a calculator and web search. "
        "Use tools when needed. When you have the answer, respond directly."
    )

    for iteration in range(max_iterations):
        response = client.messages.create(
            model="claude-3-5-haiku-20241022",
            max_tokens=1024,
            system=system,
            tools=TOOLS,
            messages=messages
        )

        if verbose:
            print(f"\n[Turn {iteration + 1}] stop_reason={response.stop_reason} "
                  f"| input_tokens={response.usage.input_tokens} "
                  f"| output_tokens={response.usage.output_tokens}")

        # Exit condition: model finished
        if response.stop_reason == "end_turn":
            for block in response.content:
                if hasattr(block, "text"):
                    return block.text
            return "(no text in final response)"

        # Tool use: dispatch and loop
        if response.stop_reason == "tool_use":
            tool_use_blocks = [b for b in response.content if b.type == "tool_use"]

            if verbose:
                for block in tool_use_blocks:
                    print(f"  -> Tool call: {block.name}({json.dumps(block.input)})")

            # Append assistant message (must include all content blocks, not just tool_use)
            messages.append({"role": "assistant", "content": response.content})

            # Execute tools and collect results
            tool_results = execute_tools(tool_use_blocks)

            if verbose:
                for r in tool_results:
                    preview = r["content"][:80].replace("\n", " ")
                    print(f"  <- Tool result [{r['tool_use_id'][:8]}...]: {preview}")

            # Append user message with tool_result blocks
            messages.append({"role": "user", "content": tool_results})
            continue

        # Unexpected stop reason (max_tokens, etc.)
        return f"Stopped with unexpected stop_reason: {response.stop_reason}"

    return f"Agent stopped after {max_iterations} iterations without reaching end_turn."


# ---------------------------------------------------------------------------
# Demo
# ---------------------------------------------------------------------------

### Demo

In [ ]:
print("=" * 60)
print("DEMO 1: Calculator tool")
print("=" * 60)
result = run_agent("What is 144 multiplied by 17, and then divided by 3?")
print(f"\nFinal answer: {result}")

print("\n" + "=" * 60)
print("DEMO 2: No tools needed (end_turn in one call)")
print("=" * 60)
result = run_agent("What are the three main components of an agent loop?")
print(f"\nFinal answer: {result}")

print("\n" + "=" * 60)
print("DEMO 3: Web search stub")
print("=" * 60)
result = run_agent("Search for the latest version of the Anthropic Python SDK and tell me what you find.")
print(f"\nFinal answer: {result}")

---

## Self-Check

Answer these without running code first:

**1. What is wrong with this approach, and what is the correct structure?**

- A. Nothing is wrong. Tool results can go in either assistant or user messages.
- B. Tool results must go in a user message as a content block with type 'tool_result', not in an assistant message.
- C. Tool results should be appended to the existing user message, not as a new message.
- D. Tool results should be sent as a system message to avoid breaking the assistant/user alternation.

**2. Which combination of changes is most likely to fix the runaway loop?**

- A. Increase max_iterations to 100 so the agent has room to finish.
- B. Remove the web_search tool entirely and use a fixed workflow instead.
- C. Update the system prompt to tell the model when one search result is sufficient, and add loop detection that exits if the same tool is called with identical inputs twice.
- D. Switch to a more powerful model that is less likely to loop.

**3. What production issue will this cause?**

- A. No issue. The API only needs the tool_use blocks in the assistant message.
- B. The API will reject the message history because the assistant message is missing required text content blocks.
- C. The model will not receive the full context of its own reasoning, which can cause it to repeat work or lose track of partial results.
- D. The token count will be slightly lower, which may cause the model to behave differently.

**4. What is the correct fix, and why does it matter?**

- A. Validate tool inputs before calling the tool and reject invalid expressions before they reach the executor.
- B. Catch the exception in the tool function and return an error string like 'Error: division by zero'. The model reads the error and decides what to do next.
- C. Catch the exception in the loop and append a tool_result with content 'null' to keep the history valid.
- D. Add a try/except in the main agent loop and restart from the beginning if any tool fails.

**5. What would you inspect to identify whether the error came from tool execution, tool dispatch, or the model's reasoning?**

- A. Only the final assistant message, since that contains the answer.
- B. The stop_reason values across all turns and the full message history, including every tool_use block and its corresponding tool_result.
- C. The model's usage stats (input_tokens and output_tokens) to see if the context was too large.
- D. The system prompt, since almost all agent errors come from poor instructions.

**6. What is the right way to handle this in production?**

- A. Always set max_iterations high enough that it never triggers.
- B. Remove max_iterations entirely so the agent can always finish.
- C. Set max_iterations based on the expected complexity of the task, log when the limit is hit, and return whatever partial result is available along with a clear indication that the task was incomplete.
- D. Catch the incomplete result and automatically restart the agent with a higher limit.

**7. How should the loop handle this, and what should the user message contain?**

- A. Execute only the first tool_use block, send that result, and wait for the model to request the second.
- B. Execute both tools in sequence, then send two separate user messages, one tool_result per message.
- C. Execute both tools, then send a single user message containing both tool_result blocks.
- D. Return an error to the model because multiple tool calls in one turn are not supported.

**8. What is the correct answer?**

- A. Yes. With streaming, tool_result blocks are sent incrementally as the stream arrives.
- B. Yes. Streaming uses a different message format where tool_use blocks are sent as events, not content blocks.
- C. No. Streaming only changes how tokens are delivered to the client. The message structure, tool dispatch, and loop logic are identical. You call stream.get_final_message() to get the same response object the non-streaming call returns.
- D. No, but you must set a different stop_reason value when using streaming to indicate tool use.

_Answers are in `checks.json` in the lesson directory._